# 01 — Domain Rental Data Curation

## Purpose
Clean and standardise the course-provided Domain.com.au Victorian rental listing dataset.

Input:
- `data/raw/vic_rentals_all.csv`

Output:
- `data/curated/vic_rentals.parquet`

This notebook:
- validates the raw schema
- checks duplicates and missingness
- standardises data types
- cleans categorical and feature fields
- flags suspicious observations
- produces a reproducible curated dataset

No raw or curated Domain listing-level data should be committed to Git.

## Imports 


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## Define Paths

In [5]:
RAW_PATH = Path("../data/raw/vic_rentals_all.csv")
CURATED_PATH = Path("../data/curated/vic_rentals.parquet")

assert RAW_PATH.exists(), f"Raw data not found: {RAW_PATH}"

CURATED_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load raw data 

In [6]:
raw = pd.read_csv(RAW_PATH)

print(f"Rows: {raw.shape[0]:,}")
print(f"Columns: {raw.shape[1]}")

Rows: 12,717
Columns: 30


## Inpsect 

In [7]:
df = raw.copy()

pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=True)
})

,dtype,missing,missing_pct,unique
listing_id,int64,0,0.00,12717
suburb,object,0,0.00,656
postcode,int64,0,0.00,426
weekly_rent,float64,279,2.19,408
bond,float64,782,6.15,983
available_date,object,136,1.07,734
date_listed,object,4,0.03,655
days_listed,float64,4,0.03,661
bedrooms,float64,125,0.98,11
bathrooms,float64,51,0.40,9


### Initial observations

- `listing_id` is complete and unique and can be used as the listing-level identifier.
- Rental and core property variables have relatively low missingness.
- `carspaces` and `structured_features` have moderate missingness.
- `property_id` has substantial missingness and is unsuitable as the primary identifier.
- `land_area` is almost entirely missing and is unlikely to be useful downstream.
- Date columns require conversion from strings to datetime.
- `postcode` should be treated as a categorical/location identifier rather than a numeric measurement.
- Latitude and longitude are almost complete, making spatial assignment to SA2 feasible.

## Validate required columns

In [8]:
required_columns = {
    "listing_id",
    "suburb",
    "postcode",
    "weekly_rent",
    "bond",
    "available_date",
    "date_listed",
    "days_listed",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "property_type",
    "address",
    "lat",
    "lon",
    "scraped_date",
    "structured_features"
}

missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Missing required columns: {missing_columns}"

## Check duplicates

In [10]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate listing IDs:", df["listing_id"].duplicated().sum())
print("Unique listing IDs:", df["listing_id"].nunique())


Exact duplicate rows: 0
Duplicate listing IDs: 0
Unique listing IDs: 12717


## Standardise text fields

In [12]:
text_columns = [
    "suburb",
    "address",
    "property_type",
    "primary_type",
    "secondary_type",
    "agency",
    "agent_names"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

## Standardise postcode

In [13]:
df["postcode"] = (
    pd.to_numeric(df["postcode"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(4)
)

## Clean numeric columns

In [14]:
numeric_columns = [
    "weekly_rent",
    "bond",
    "days_listed",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "photo_count",
    "video_count",
    "floorplans_count",
    "lat",
    "lon"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

## Convert integer-like fields

In [15]:
integer_columns = [
    "bedrooms",
    "bathrooms",
    "carspaces",
    "photo_count",
    "video_count",
    "floorplans_count",
    "days_listed"
]

for col in integer_columns:
    if col in df.columns:
        df[col] = df[col].round().astype("Int64")

## Parse dates